# 04. Dublery testowe i gMock

## Cele

Po tym notebooku potrafisz:

* rozpoznać zależności, które utrudniają testowanie, i odciąć je przez interfejs (wstrzykiwanie zależności),
* rozróżnić rodzaje dublerów testowych: dummy, stub, fake, spy, mock,
* napisać mock w gMock (`MOCK_METHOD`) i wyjaśnić, dlaczego mockowana metoda musi być wirtualna,
* używać `ON_CALL` i `EXPECT_CALL` z akcjami (`Return`, `Throw`), krotnościami (`Times`) i matcherami,
* przewidzieć, które oczekiwanie gMock dopasuje, gdy jest ich kilka.

## Problem: zależności

Testowana klasa rzadko żyje sama. Korzysta z sieci, bazy danych, systemu plików, zegara czy generatora
liczb losowych. Test, który używa ich naprawdę:

* jest **wolny** (sieć, dysk) — łamie zasadę *Fast*,
* jest **niepowtarzalny** — wynik zależy od stanu serwera czy losowania (*Repeatable*),
* **nie potrafi wywołać przypadków brzegowych** — jak sprawdzić reakcję na awarię serwera, jeśli serwer działa?

Rozwiązaniem jest podstawienie w teście **dublera** (*test double*) — obiektu, który udaje prawdziwą
zależność ([M. Fowler, *TestDouble*](https://martinfowler.com/bliki/TestDouble.html);
nazwa pochodzi od dublera kaskadera w filmie).

## Rodzaje dublerów

Klasyfikacja G. Meszarosa (*xUnit Test Patterns*), w skrócie M. Fowlera:

| Dubler | Co robi | Przykład |
|---|---|---|
| **Dummy** | jest przekazywany, ale nigdy nieużywany — tylko „wypełnia” listę parametrów | pusty obiekt loggera wymagany przez konstruktor |
| **Stub** | zwraca przygotowane odpowiedzi | „kurs EUR zawsze wynosi 4,30” |
| **Spy** | stub, który dodatkowo zapamiętuje, jak był wywoływany — test sprawdza to po fakcie | zapisuje treść wysłanych e-maili |
| **Mock** | ma zaprogramowane **oczekiwania** co do wywołań i sam sprawdza, czy zostały spełnione | „`charge` musi zostać wywołane dokładnie raz z kwotą 1500” |
| **Fake** | ma działającą, uproszczoną implementację, nienadającą się na produkcję | baza danych w pamięci (`std::map` zamiast SQL) |

Stub i fake służą do **sterowania wejściem** testowanego kodu (test sprawdza potem stan albo wynik).
Mock służy do **weryfikacji zachowania** — tego, *czy i jak* kod wywołał zależność
([M. Fowler, *Mocks Aren't Stubs*](https://martinfowler.com/articles/mocksArentStubs.html)).
Po polsku mock bywa nazywany *atrapą obiektu* ([Wikipedia](https://pl.wikipedia.org/wiki/Atrapa_obiektu)).

## Wstrzykiwanie zależności przez interfejs

Żeby podstawić dublera, testowana klasa nie może sama tworzyć zależności ani wołać jej „na sztywno”.
Zależność opisujemy **interfejsem** (klasa abstrakcyjna z metodami czysto wirtualnymi i wirtualnym
destruktorem), a obiekt przekazujemy z zewnątrz — najczęściej w konstruktorze. To jest
**wstrzykiwanie zależności** (*dependency injection*). Kod produkcyjny przekazuje prawdziwą
implementację, a test — dublera.

M. Feathers nazywa takie miejsce **szwem** (*seam*): to miejsce, w którym można zmienić zachowanie
programu bez edytowania kodu w tym miejscu. Szew obiektowy działa dzięki wywołaniom metod wirtualnych
([*Working Effectively with Legacy Code*, rozdz. 4 — fragment](https://www.informit.com/articles/article.aspx?p=359417&seqNum=2);
[SWE at Google, rozdz. 13 — Seams](https://abseil.io/resources/swe-book/html/ch13.html#seams)).

### Przykład: wstrzyknięty zegar

Kod, który sam odczytuje bieżący czas, trudno przetestować. Wystarczy wydzielić
interfejs zegara i wstrzyknąć go — w teście podstawiamy ręcznie napisany dubler z ustawioną godziną:

```cpp
#include <string>

#include <gtest/gtest.h>

class Clock {
public:
    virtual ~Clock() = default;
    virtual int currentHour() const = 0;  // 0-23
};

class Greeter {
public:
    explicit Greeter(const Clock& clock) : clock_(clock) {}

    std::string greeting() const {
        return clock_.currentHour() < 18 ? "Dzień dobry" : "Dobry wieczór";
    }

private:
    const Clock& clock_;
};

// Test double with a fixed time: the test fully controls "now".
class FixedClock : public Clock {
public:
    explicit FixedClock(int hour) : hour_(hour) {}
    int currentHour() const override { return hour_; }

private:
    int hour_;
};

TEST(GreeterTest, DayGreetingBefore18) {
    FixedClock clock(17);
    EXPECT_EQ(Greeter(clock).greeting(), "Dzień dobry");
}

TEST(GreeterTest, EveningGreetingFrom18) {
    FixedClock clock(18);
    EXPECT_EQ(Greeter(clock).greeting(), "Dobry wieczór");
}
```

W programie przekazalibyśmy klasę `SystemClock`, której `currentHour()` korzysta z `std::chrono::system_clock::now()`.
Testy przechodzą o każdej porze dnia i sprawdzają granicę 18:00, na którą w rzeczywistości trzeba by czekać.
Więcej o testowaniu kodu zależnego od czasu w C++: [S. Dargo, *Time in C++: Once More About Testing*](https://www.sandordargo.com/blog/2026/01/28/clocks-part-9-once-more-about-testing).

## gMock: mock w jednej linii

Ręczne dublery są dobre dla prostych przypadków. Gdy chcemy sterować odpowiedziami w każdym teście
inaczej albo sprawdzać wywołania, wygodniej użyć **gMock** — części GoogleTest
([gMock for Dummies](https://google.github.io/googletest/gmock_for_dummies.html)).

Klasa mocka dziedziczy po interfejsie i dla każdej metody zawiera makro
`MOCK_METHOD(typ_zwracany, nazwa, (argumenty), (specyfikatory))`:

```cpp
class PaymentGateway {
public:
    virtual ~PaymentGateway() = default;
    virtual bool charge(const std::string& customerId, int amountCents) = 0;
};

class MockPaymentGateway : public PaymentGateway {
public:
    MOCK_METHOD(bool, charge, (const std::string& customerId, int amountCents), (override));
};
```

* Argumenty podajemy w nawiasach. Jeśli typ zawiera przecinek (np. `std::map<int, int>`), też trzeba go
  objąć nawiasami.
* Specyfikatory: `override` (zalecany), `const` (wymagany, gdy mockowana metoda jest `const`), np. `(const, override)`
  ([Cookbook — Creating Mock Classes](https://google.github.io/googletest/gmock_cook_book.html#creating-mock-classes)).
* Program testowy linkujemy z `GTest::gmock_main` zamiast `GTest::gtest_main`.

## Dlaczego mockowana metoda musi być wirtualna?

Testowany kod zna tylko interfejs — przechowuje referencję `PaymentGateway&`. Wywołanie przez taką
referencję trafia do implementacji z klasy pochodnej (mocka) **tylko wtedy, gdy metoda jest wirtualna**
([cppreference — virtual](https://en.cppreference.com/w/cpp/language/virtual)).

Jeśli metoda w klasie bazowej **nie jest** wirtualna, a `MOCK_METHOD` napiszemy bez `override`,
kod nadal się skompiluje: metoda mocka jedynie *przesłania* (ukrywa) metodę bazową o tej samej nazwie.
Kod testowany wywołujący ją przez referencję do klasy bazowej trafi jednak do **prawdziwej**
implementacji, a mock nie zobaczy żadnego wywołania — test zachowa się inaczej, niż zakładaliśmy,
a jego oczekiwania nie zostaną spełnione.

Dlatego warto zawsze pisać `(override)`: kompilator sprawdzi wtedy, że metoda rzeczywiście nadpisuje
metodę wirtualną, i zgłosi błąd zamiast cichej pomyłki
([cppreference — override](https://en.cppreference.com/w/cpp/language/override)):

```cpp
#include <gmock/gmock.h>

class Thermometer {
public:
    int readCelsius();  // NOT virtual
};

class MockThermometer : public Thermometer {
public:
    // error: 'readCelsius' marked 'override', but does not override
    MOCK_METHOD(int, readCelsius, (), (override));
};
```

gMock potrafi mockować metody niewirtualne, ale wymaga to szablonów — testowany kod musi wtedy
przyjmować typ zależności jako parametr szablonu, a wybór następuje w czasie kompilacji
([Cookbook — Mocking Non-virtual Methods](https://google.github.io/googletest/gmock_cook_book.html#MockingNonVirtualMethods)).

## `ON_CALL` i `EXPECT_CALL`

| | `ON_CALL(mock, metoda(...))` | `EXPECT_CALL(mock, metoda(...))` |
|---|---|---|
| określa, co metoda zwraca | tak — `.WillByDefault(akcja)` | tak — `.WillOnce(...)`, `.WillRepeatedly(...)` |
| wymaga, żeby metoda została wywołana | **nie** | **tak** — określoną liczbę razy |
| rola dublera | stub | mock |

Dokumentacja zaleca: **domyślnie `ON_CALL`, a `EXPECT_CALL` tylko wtedy, gdy celem testu jest
sprawdzenie samego wywołania** — każde zbędne oczekiwanie usztywnia test
([Cookbook — Knowing When to Expect](https://google.github.io/googletest/gmock_cook_book.html#UseOnCall)).

Ważne zasady ([gMock for Dummies — Using Mocks in Tests](https://google.github.io/googletest/gmock_for_dummies.html#using-mocks-in-tests)):

* oczekiwania ustawiamy **przed** wywołaniami metod mocka,
* gMock sprawdza, czy wszystkie oczekiwania zostały spełnione, **w chwili niszczenia mocka** —
  zwykle na końcu testu. Niespełnione `EXPECT_CALL` (np. metoda nie została wywołana) oznacza nieudany test.

## Akcje, krotności i matchery

**Akcje** — co ma zrobić wywołana metoda ([Actions Reference](https://google.github.io/googletest/reference/actions.html)):
`Return(wartość)`, `Throw(wyjątek)`, `Return()` dla `void`.
`Return` przyjmuje każdą wartość, którą da się zamienić na typ zwracany metody — np. dla
`std::optional<int>` zarówno `Return(5)`, jak i `Return(std::nullopt)`.
`.WillOnce(a)` wykonuje akcję przy jednym wywołaniu (można dopisać kilka), `.WillRepeatedly(a)` — przy
wszystkich kolejnych.

**Krotność** — ile razy metoda ma być wywołana: `.Times(n)`, `.Times(0)` (nie może być wywołana),
`.Times(::testing::AtLeast(n))`, `.Times(::testing::AnyNumber())`.
Jeśli `Times` pominiemy, gMock wyznacza ją sam
([gMock for Dummies — Cardinalities](https://google.github.io/googletest/gmock_for_dummies.html#cardinalities-how-many-times-will-it-be-called)):

* bez `WillOnce` i `WillRepeatedly` → `Times(1)`,
* *n* × `WillOnce` → `Times(n)`,
* *n* × `WillOnce` + `WillRepeatedly` → `Times(AtLeast(n))`.

Gdy wszystkie `WillOnce` się wyczerpią, a nie ma `WillRepeatedly`, metoda wykonuje akcję domyślną
(dla `int` zwraca 0, dla `bool` — `false`).

**Matchery** — jakie argumenty pasują do oczekiwania: konkretna wartość (`charge("alice", 1500)`),
`_` (cokolwiek), `Gt(0)`, `Lt(n)`, `Ne(x)`, `HasSubstr("...")` i wiele innych
([Matchers Reference](https://google.github.io/googletest/reference/matchers.html)).
Zapis `EXPECT_CALL(mock, charge)` bez nawiasów oznacza dowolne argumenty.

```cpp
using ::testing::_;
using ::testing::Gt;
using ::testing::Return;

EXPECT_CALL(gateway, charge("alice", Gt(0)))
    .Times(2)
    .WillOnce(Return(true))
    .WillOnce(Return(false));
EXPECT_CALL(gateway, charge("bob", _)).Times(0);
```

## Demonstracja

Plik [`demos/order_service_test.cpp`](demos/order_service_test.cpp) zawiera klasę `OrderService`
(zamówienie → obciążenie karty przez `PaymentGateway`) i cztery testy: stub (`ON_CALL`), weryfikację
wywołania (`EXPECT_CALL ... Times(1)`), brak wywołania (`Times(0)`) i symulację awarii (`Throw`).

In [ ]:
# Build and run the gMock demo
!cmake -S demos -B demos/build && cmake --build demos/build --config Debug --parallel && ctest --test-dir demos/build -C Debug --output-on-failure

Spróbuj: w `OrderService::placeOrder` usuń warunek `amountCents <= 0`, zapisz plik i uruchom komórkę
ponownie. Który test nie przechodzi i co mówi komunikat gMock?

## Kilka oczekiwań dla jednej metody

Gdy dla tej samej metody zdefiniujemy kilka `EXPECT_CALL`, gMock przy każdym wywołaniu przeszukuje
je **od najnowszego do najstarszego** i wybiera pierwsze aktywne oczekiwanie pasujące do argumentów
(„nowsze reguły nadpisują starsze”). Dzięki temu można ustawić ogólne oczekiwanie (np. w fixture),
a w teście dopisać bardziej szczegółowe
([gMock for Dummies — Using Multiple Expectations](https://google.github.io/googletest/gmock_for_dummies.html#MultiExpectations)).

Oczekiwania są domyślnie **„lepkie”** (*sticky*): pozostają aktywne także po osiągnięciu górnego limitu
wywołań. Kolejne pasujące wywołanie nie przejdzie więc do starszego oczekiwania, tylko przekroczy limit
nowszego i spowoduje błąd ([All Expectations Are Sticky](https://google.github.io/googletest/gmock_for_dummies.html#StickyExpectations)).
Jeśli oczekiwanie ma „wygasnąć” po wykorzystaniu, dopisujemy `.RetiresOnSaturation()`
([Mocking Reference](https://google.github.io/googletest/reference/mocking.html#EXPECT_CALL.RetiresOnSaturation)):

```cpp
#include <gmock/gmock.h>
#include <gtest/gtest.h>

class Counter {
public:
    virtual ~Counter() = default;
    virtual int next(int step) = 0;
};

class MockCounter : public Counter {
public:
    MOCK_METHOD(int, next, (int step), (override));
};

using ::testing::_;
using ::testing::Return;

TEST(ExpectationOrderDemo, NewerSpecificExpectationRetires) {
    MockCounter counter;
    EXPECT_CALL(counter, next(_)).WillRepeatedly(Return(0));  // general, older
    EXPECT_CALL(counter, next(5))                             // specific, newer
        .WillOnce(Return(100))
        .RetiresOnSaturation();

    EXPECT_EQ(counter.next(5), 100);  // newer expectation matches first
    EXPECT_EQ(counter.next(5), 0);    // it retired, so the older one is used
    EXPECT_EQ(counter.next(1), 0);
}
```

Bez `.RetiresOnSaturation()` drugie wywołanie `next(5)` nadal pasowałoby do nowszego oczekiwania
i przekroczyłoby jego limit (jedno `WillOnce` → `Times(1)`) — test by nie przeszedł.
Jeśli zależy nam na kolejności wywołań, służy do tego `::testing::InSequence`
([Ordered vs Unordered Calls](https://google.github.io/googletest/gmock_for_dummies.html#OrderedCalls)).

## Wywołania „nieciekawe”: `NiceMock` i `StrictMock`

Wywołanie metody, dla której nie ma żadnego `EXPECT_CALL`, to wywołanie **nieciekawe** (*uninteresting*):
gMock wykonuje akcję domyślną (lub tę z `ON_CALL`) i wypisuje ostrzeżenie, ale test przechodzi.
Wywołanie, które nie pasuje do żadnego z istniejących `EXPECT_CALL` tej metody, to wywołanie
**nieoczekiwane** (*unexpected*) — zawsze błąd.

* `::testing::NiceMock<MockX>` — wycisza ostrzeżenia o nieciekawych wywołaniach,
* `::testing::StrictMock<MockX>` — traktuje je jak błędy.

Dokumentacja ostrzega, że ścisłe mocki czynią testy kruchymi
([Cookbook — The Nice, the Strict, and the Naggy](https://google.github.io/googletest/gmock_cook_book.html#NiceStrictNaggy)).

## Nie przesadzaj z mockami

Mock sprawdza, *jak* kod rozmawia z zależnością, więc wiąże test ze szczegółami implementacji.
Zbyt wiele `EXPECT_CALL` sprawia, że testy psują się przy każdej refaktoryzacji, choć zachowanie się nie
zmieniło. Gdy prawdziwy obiekt jest szybki i deterministyczny, użyj go; gdy nie — rozważ fake;
mocka używaj tam, gdzie interakcja jest sednem wymagania (np. „karta zostaje obciążona dokładnie raz”)
([Google Testing Blog — *Don't Overuse Mocks*](https://testing.googleblog.com/2013/05/testing-on-toilet-dont-overuse-mocks.html);
[*Increase Test Fidelity By Avoiding Mocks*](https://testing.googleblog.com/2024/02/increase-test-fidelity-by-avoiding-mocks.html)).

## Ćwiczenie 4 — `WeatherService` i gMock

Aplikacja ostrzega przed przymrozkami. Klasa `WeatherService` ([`src/weather_service.h`](src/weather_service.h))
pobiera temperaturę z zewnętrznej usługi przez interfejs `IWeatherApi` ([`src/weather_api.h`](src/weather_api.h)):

```cpp
class IWeatherApi {
public:
    virtual ~IWeatherApi() = default;
    virtual std::optional<int> fetchTemperature(int cityId) = 0;  // std::nullopt when the request fails
};
```

Zachowanie `WeatherService::frostStatus(cityId)`:

* temperatura **≤ 0 °C** → `FrostStatus::Frost`, powyżej 0 °C → `FrostStatus::NoFrost`,
* nieudane zapytanie (`std::nullopt`) → `FrostStatus::Unknown`,
* udany odczyt jest zapamiętywany: dla danego miasta usługa jest odpytywana **co najwyżej raz**
  (nieudane zapytania nie są zapamiętywane).

W teście nie chcemy łączyć się z prawdziwą usługą — podstawimy mock.

**Zadanie.**

1. W pliku <b>[tests/mock_weather_api.h](tests/mock_weather_api.h)</b> napisz klasę `MockWeatherApi`
   — mock interfejsu `IWeatherApi` (`MOCK_METHOD` z typem zwracanym `std::optional<int>`).
2. W pliku <b>[tests/test_weather_service.cpp](tests/test_weather_service.cpp)</b> zastąp każde `FAIL()`
   testem:
   * `ReportsFrostBelowZero`, `ReportsNoFrostAboveZero`, `ZeroDegreesIsFrost` — mock zwraca −5, 3 i 0
     (wartość brzegowa!); sprawdź wynik `frostStatus`,
   * `ReportsUnknownWhenApiFails` — zapytanie się nie udaje (`std::nullopt`) → `Unknown`,
   * `AsksApiOnlyOncePerCity` — dwa wywołania `frostStatus(1)`, a `fetchTemperature(1)` ma być wywołane
     **dokładnie raz**,
   * `AsksApiForEachCity` — `frostStatus(1)` i `frostStatus(2)` odpytują usługę osobno dla każdego miasta.

<details>
<summary><b>Wskazówki</b></summary>

* W pierwszych czterech testach liczba wywołań nie jest sednem testu — wystarczy `ON_CALL` (stub).
  Ostrzeżenia o „uninteresting call” wyciszysz, tworząc mock jako `::testing::NiceMock<MockWeatherApi>`.
* Nieudane zapytanie zasymulujesz akcją `Return(std::nullopt)`.
* W dwóch ostatnich testach liczba wywołań **jest** wymaganiem, więc to miejsce na `EXPECT_CALL`
  z `.Times(1)` — osobno dla każdego miasta.
* Test dla chętnych: dwa `WillOnce` w jednym `EXPECT_CALL`.

</details>

### Zbuduj i uruchom testy

In [ ]:
# Configure, build and run the exercise tests
!cmake -S . -B build && cmake --build build --config Debug --parallel && ctest --test-dir build -C Debug --output-on-failure

Przed zmianami sześć testów nie przechodzi z komunikatem `TODO`. Po wykonaniu zadania:

```
100% tests passed, 0 tests failed out of 6
```

Sprawdź, czy test pamięci podręcznej naprawdę coś sprawdza: w `WeatherService::frostStatus`
([`src/weather_service.cpp`](src/weather_service.cpp)) tymczasowo zakomentuj zapisywanie odczytu
do `temperatureByCity_` i uruchom testy ponownie. Który test nie przechodzi i co pisze gMock?
Potem przywróć kod.

**Dla chętnych:** napisz test `RetriesAfterFailure` — pierwsze zapytanie o miasto kończy się
niepowodzeniem (`Unknown`), drugie zwraca 4 °C (`NoFrost`).

## Linki i materiały

* [gMock for Dummies](https://google.github.io/googletest/gmock_for_dummies.html) — wprowadzenie
* [gMock Cookbook](https://google.github.io/googletest/gmock_cook_book.html) — przepisy na konkretne problemy
* [gMock Cheat Sheet](https://google.github.io/googletest/gmock_cheat_sheet.html) — ściąga
* [M. Fowler — *Mocks Aren't Stubs*](https://martinfowler.com/articles/mocksArentStubs.html)
* [Software Engineering at Google, rozdz. 13 — *Test Doubles*](https://abseil.io/resources/swe-book/html/ch13.html)
* [Google Testing Blog — *Know Your Test Doubles*](https://testing.googleblog.com/2013/07/testing-on-toilet-know-your-test-doubles.html)
* [M. Hargett — *Advanced Unit Testing in C & C++*, CppCon 2015 (wideo)](https://www.youtube.com/watch?v=Wmy6g-aVgZI)

---

[← 03. Fixture'y i cykl życia testu](../03_fixtures/03_fixtures.ipynb) | [Strona główna](../_start.ipynb) | [05. Jakość testów →](../05_quality/05_quality.ipynb)